# The WFDB library API

Besides the high-level functions (`read_record`, `read_annotations`, ...),
wfdb-x provides every function of the C WFDB library under its C name, with
the same arguments and return values. Where a C function fills an array
supplied by the caller, the Python function fills a list. Errors are
reported as in C: a negative return value, and a message (also available
from `wfdberror()`).

In [ ]:
import numpy as np
import wfdbx

# Look for records in the current directory, then in the sample data shipped
# with wfdb-x (the record "100s").
wfdbx.setwfdb(". " + wfdbx.sample_data_dir())

## Signals

`isigopen` opens the signals of a record and fills a list of `Siginfo`;
`getvec` reads one sample of each signal; `isigsettime` skips to a time.

In [ ]:
si = []
nsig = wfdbx.isigopen("100s", si, 2)
print(nsig, "signals:", [(s.desc, s.fmt, s.gain) for s in si])
print("sampling frequency:", wfdbx.sampfreq(None))

v = [0] * nsig
for _ in range(3):
    wfdbx.getvec(v)
    print(v)

In [ ]:
t = wfdbx.strtim("0:30")              # a time string -> sample number
print("0:30 is sample", t, "=", wfdbx.mstimstr(t))
wfdbx.isigsettime(t)
wfdbx.getvec(v)
print("samples at 0:30:", v, "->", [wfdbx.aduphys(k, v[k]) for k in range(nsig)])

`getframes(n)` (an addition of wfdb-x) reads up to `n` sample vectors at once
into a NumPy array, exactly as `n` calls of `getvec` would. It also returns
the `getvec` status that ended the reading (the number of signals if all `n`
were read):

In [ ]:
frames, stat = wfdbx.getframes(360)
print(frames.shape, stat)
print(frames[:2])

## Annotations

In [ ]:
a = wfdbx.Annotation()
wfdbx.annopen("100s", [wfdbx.Anninfo("atr", wfdbx.WFDB_READ)], 1)
for _ in range(5):
    wfdbx.getann(0, a)
    print(wfdbx.mstimstr(a.time), wfdbx.annstr(a.anntyp), a.aux_string() or "")

## Errors and cleaning up

In [ ]:
wfdbx.wfdbquiet()                        # no messages on stderr
print(wfdbx.isigopen("nosuchrecord", [], 2), "-", wfdbx.wfdberror().strip())
wfdbx.wfdbverbose()
wfdbx.wfdbquit()                         # close all files, reset the state